# Spectral Wristband Benchmark

Comparison of latent-space regularization methods.

**Methods** (6 total):
- **Spectral Wristband AE** (ours) — O(NdK) spectral decomposition
- **Pairwise Wristband AE** — O(N²d) pairwise kernel (ablation)
- **WAE-MMD** — Wasserstein AE with IMQ kernel
- **VAE** — Variational AE with KL divergence
- **SWAE** — Sliced Wasserstein AE
- **RFF-MMD AE** — Random Fourier Features MMD

**Metrics**: FID ↓, Reconstruction MSE ↓, Step time (ms) ↓, Peak memory (MB) ↓

**Runtime**: GPU required (T4 or A100) for timing results.

## 0. Setup

In [1]:
# Install dependencies (Colab)
# Torch is pre-installed on Colab; we just add pythae + pytorch-fid
!pip install -q pythae pytorch-fid scipy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.0/235.0 kB 13.1 MB/s eta 0:00:00


In [2]:
# Clone repos (Colab only — skip if running locally)
import os, sys

BRANCH = "paper-method-benchmark"

if 'google.colab' in sys.modules:
    REPO = '/content/wristband-loss-proofs'
    TIDBITS = '/content/ml-tidbits'
    if not os.path.exists(REPO):
        !git clone -b {BRANCH} https://github.com/andremiguelc/wristband-loss-proofs.git {REPO}
    if not os.path.exists(TIDBITS):
        !git clone https://github.com/mvparakhin/ml-tidbits.git {TIDBITS}
    os.chdir(os.path.join(REPO, 'python', 'benchmark'))
else:
    # Local: notebook lives at python/benchmark/
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    TIDBITS = os.path.join(REPO, 'ml-tidbits')

# Add to Python path
sys.path.insert(0, os.path.join(REPO, 'python'))
sys.path.insert(0, os.path.join(TIDBITS, 'python'))

print(f"REPO: {REPO}")
print(f"ML-TIDBITS: {TIDBITS}")
print(f"Branch: {BRANCH}")
print(f"Python: {sys.version}")
print(f"CWD: {os.getcwd()}")

REPO: /content/wristband-loss-proofs
ML-TIDBITS: /content/ml-tidbits
Branch: paper-method-benchmark
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CWD: /content/wristband-loss-proofs/python/benchmark


In [ ]:
import time
import copy
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from pythae.models import AEConfig, VAE, VAEConfig
from pythae.models.wae_mmd import WAE_MMD, WAE_MMD_Config
from pythae.models.base.base_utils import ModelOutput
from pythae.models.nn.benchmarks.mnist import (
    Encoder_Conv_AE_MNIST,
    Decoder_Conv_AE_MNIST,
    Encoder_Conv_VAE_MNIST,
)

from benchmark.models import (
    SpectralWristbandAE, SpectralWristbandConfig,
    PairwiseWristbandAE, PairwiseWristbandConfig,
    SWAE, SWAEConfig,
    RFF_MMD_AE, RFF_MMD_Config,
)

# Device + reproducibility
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

Device: cuda
GPU: Tesla T4
CUDA: 12.8
PyTorch: 2.10.0+cu128


## 1. Data

Load MNIST, normalize to [0, 1], reshape to (N, 1, 28, 28).

In [ ]:
# --- Configuration ---
DATASET = "mnist"
LATENT_DIM = 16
BATCH_SIZE = 256
NUM_EPOCHS = 5          # quick sanity test; increase for final runs
LEARNING_RATE = 1e-3
INPUT_DIM = (1, 28, 28)

# --- Load MNIST ---
transform = torchvision.transforms.ToTensor()  # scales to [0, 1]
train_ds = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=transform)
eval_ds = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Dataset wrapper yielding {"data": tensor} dicts (expected by our model forward())
from torch.utils.data import Dataset as TorchDataset

class ImageDictDataset(TorchDataset):
    """Wraps a torchvision dataset into {"data": tensor} format."""
    def __init__(self, tv_dataset):
        self.imgs = torch.stack([tv_dataset[i][0] for i in range(len(tv_dataset))])
    def __len__(self):
        return len(self.imgs)
    def __getitem__(self, idx):
        return {"data": self.imgs[idx]}

train_data = ImageDictDataset(train_ds)
eval_data = ImageDictDataset(eval_ds)

print(f"Train: {len(train_data)} samples, Eval: {len(eval_data)} samples")
print(f"Sample shape: {train_data[0]['data'].shape}")
print(f"Range: [{train_data.imgs.min():.2f}, {train_data.imgs.max():.2f}]")

Train: 60000 samples, Eval: 10000 samples
Sample shape: torch.Size([1, 28, 28])
Range: [0.00, 1.00]


## 2. Model Definitions

Define all 6 methods. VAE and WAE-MMD use Pythae model classes; the other 4 use custom subclasses from `models.py`. All share the same CNN encoder/decoder architecture.

In [ ]:
def make_models(latent_dim: int, input_dim: tuple) -> dict:
    """Create all 6 models with shared encoder/decoder architectures.

    Returns dict of {name: model}.
    """
    models = {}

    # --- 1. Spectral Wristband AE (ours) ---
    cfg = SpectralWristbandConfig(
        input_dim=input_dim, latent_dim=latent_dim,
        reg_weight=1.0, beta=8.0, k_modes=6,
    )
    models["Spectral Wristband"] = SpectralWristbandAE(
        model_config=cfg,
        encoder=Encoder_Conv_AE_MNIST(cfg),
        decoder=Decoder_Conv_AE_MNIST(cfg),
    )

    # --- 2. Pairwise Wristband AE (ablation) ---
    cfg = PairwiseWristbandConfig(
        input_dim=input_dim, latent_dim=latent_dim,
        reg_weight=1.0, beta=8.0,
    )
    models["Pairwise Wristband"] = PairwiseWristbandAE(
        model_config=cfg,
        encoder=Encoder_Conv_AE_MNIST(cfg),
        decoder=Decoder_Conv_AE_MNIST(cfg),
    )

    # --- 3. WAE-MMD (Pythae built-in) ---
    cfg = WAE_MMD_Config(
        input_dim=input_dim, latent_dim=latent_dim,
        kernel_choice="imq", reg_weight=100.0,
    )
    models["WAE-MMD"] = WAE_MMD(
        model_config=cfg,
        encoder=Encoder_Conv_AE_MNIST(cfg),
        decoder=Decoder_Conv_AE_MNIST(cfg),
    )

    # --- 4. VAE (Pythae built-in) ---
    cfg = VAEConfig(input_dim=input_dim, latent_dim=latent_dim)
    models["VAE"] = VAE(
        model_config=cfg,
        encoder=Encoder_Conv_VAE_MNIST(cfg),
        decoder=Decoder_Conv_AE_MNIST(cfg),
    )

    # --- 5. SWAE ---
    cfg = SWAEConfig(
        input_dim=input_dim, latent_dim=latent_dim,
        reg_weight=10.0, num_projections=50,
    )
    models["SWAE"] = SWAE(
        model_config=cfg,
        encoder=Encoder_Conv_AE_MNIST(cfg),
        decoder=Decoder_Conv_AE_MNIST(cfg),
    )

    # --- 6. RFF-MMD AE ---
    cfg = RFF_MMD_Config(
        input_dim=input_dim, latent_dim=latent_dim,
        reg_weight=10.0, rff_dim=500,
    )
    models["RFF-MMD"] = RFF_MMD_AE(
        model_config=cfg,
        encoder=Encoder_Conv_AE_MNIST(cfg),
        decoder=Decoder_Conv_AE_MNIST(cfg),
    )

    return models

models = make_models(LATENT_DIM, INPUT_DIM)
print(f"Models: {list(models.keys())}")

## 3. Training

Plain PyTorch training loop with per-batch timing (cuda.synchronize) and per-model memory tracking.

In [ ]:
from tqdm.auto import tqdm

def train_model(name: str, model, train_data, eval_data):
    """Train one model with a plain PyTorch loop. Return metrics dict."""
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=(DEVICE == "cuda"))
    eval_loader = DataLoader(eval_data, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0)

    train_losses, eval_recon_losses, step_times = [], [], []

    # Measure memory baseline AFTER model + optimizer are on GPU,
    # so we only report the incremental training cost.
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        # One dummy step to trigger lazy CUDA allocations (cuDNN plans, etc.)
        with torch.no_grad():
            _x = torch.rand(2, *INPUT_DIM, device=DEVICE)
            _o = model({"data": _x})
        del _x, _o
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        mem_baseline = torch.cuda.memory_allocated()

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f"  Epoch {epoch}/{NUM_EPOCHS}",
                    leave=False, unit="batch")
        for batch in pbar:
            x = batch["data"].float().to(DEVICE)

            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            out = model({"data": x})
            optimizer.zero_grad()
            out.loss.backward()
            optimizer.step()

            if DEVICE == "cuda":
                torch.cuda.synchronize()
            dt_ms = (time.perf_counter() - t0) * 1000
            step_times.append(dt_ms)

            epoch_loss += out.loss.item()
            n_batches += 1
            pbar.set_postfix(loss=f"{out.loss.item():.4f}", ms=f"{dt_ms:.1f}")

        avg_train = epoch_loss / n_batches
        train_losses.append(avg_train)

        # Eval pass — track reconstruction loss only (comparable across methods,
        # not distorted by different reg scales or BatchNorm train/eval shift)
        model.eval()
        eval_recon = 0.0
        n_eval = 0
        with torch.no_grad():
            for batch in eval_loader:
                x = batch["data"].float().to(DEVICE)
                out = model({"data": x})
                # Compute recon MSE in the same scale as training (sum over pixels, mean over batch)
                recon = F.mse_loss(
                    out.recon_x.reshape(x.shape[0], -1),
                    x.reshape(x.shape[0], -1),
                    reduction="none",
                ).sum(dim=-1).mean()
                eval_recon += recon.item()
                n_eval += 1
        avg_eval_recon = eval_recon / n_eval
        eval_recon_losses.append(avg_eval_recon)

        recent_ms = np.mean(step_times[-n_batches:])
        print(f"  Epoch {epoch}/{NUM_EPOCHS}  train={avg_train:.4f}  "
              f"eval_recon={avg_eval_recon:.4f}  step={recent_ms:.1f}ms")

    peak_mem = 0.0
    if DEVICE == "cuda":
        # Peak minus baseline = incremental memory used by training
        peak_mem = (torch.cuda.max_memory_allocated() - mem_baseline) / (1024 ** 2)

    # Final eval: reconstruction MSE + latent diagnostics
    model.eval()
    with torch.no_grad():
        recon_mses, z_all = [], []
        for i in range(0, len(eval_data.imgs), BATCH_SIZE):
            batch = eval_data.imgs[i:i+BATCH_SIZE].float().to(DEVICE)
            out = model({"data": batch})
            recon_mse = F.mse_loss(
                out.recon_x.reshape(batch.shape[0], -1),
                batch.reshape(batch.shape[0], -1),
            )
            recon_mses.append(recon_mse.item())
            z_all.append(out.z.cpu())
        z_all = torch.cat(z_all, dim=0)

    metrics = {
        "name": name,
        "mean_step_ms": float(np.mean(step_times)),
        "peak_memory_mb": peak_mem,
        "train_losses": train_losses,
        "eval_recon_losses": eval_recon_losses,
        "step_times_ms": step_times,
        "recon_mse": float(np.mean(recon_mses)),
        "latent_mean_norm": float(z_all.mean(0).norm()),
        "latent_cov_frob": float(
            (torch.cov(z_all.T) - torch.eye(LATENT_DIM)).norm()
        ),
    }

    print(f"  Step time: {metrics['mean_step_ms']:.1f} ms | "
          f"Recon MSE: {metrics['recon_mse']:.6f}")
    print(f"  ||mu||: {metrics['latent_mean_norm']:.4f} | "
          f"||S-I||_F: {metrics['latent_cov_frob']:.4f}")
    if peak_mem > 0:
        print(f"  Peak memory: {peak_mem:.1f} MB")

    return metrics

In [ ]:
# Train all models
all_metrics = {}
model_names = list(models.keys())
for i, name in enumerate(model_names, 1):
    print(f"\n[{i}/{len(model_names)}]", end="")
    torch.manual_seed(42)  # reset seed for each model
    metrics = train_model(name, models[name], train_data, eval_data)
    all_metrics[name] = metrics

print(f"\n{'='*60}")
print("All models trained.")
print(f"{'='*60}")


[1/6]
Training: Spectral Wristband


  Epoch 1/5:   0%|          | 0/235 [00:00<?, ?batch/s]

  Epoch 1/5  train=8.5402  eval=42.5136  step=179.1ms


  Epoch 2/5:   0%|          | 0/235 [00:00<?, ?batch/s]

  Epoch 2/5  train=6.1019  eval=38.8109  step=166.4ms


...

  Epoch 3/5  train=5.9047  eval=5.5763  step=167.1ms


  Epoch 4/5:   0%|          | 0/235 [00:00<?, ?batch/s]

  Epoch 4/5  train=5.6245  eval=5.3420  step=167.1ms


  Epoch 5/5:   0%|          | 0/235 [00:00<?, ?batch/s]

  Epoch 5/5  train=5.3365  eval=5.2899  step=167.2ms
  Step time: 167.0 ms | Recon MSE: 0.006732
  ||mu||: 4.4225 | ||S-I||_F: 52.4897
  Peak memory: 434.8 MB

All models trained.


## 4. Results

In [ ]:
# --- Results Table ---
print(f"\n{'Method':<22} {'Recon MSE':>10} {'Step ms':>10} {'Mem MB':>10} {'||mu||':>10} {'||S-I||_F':>10}")
print("-" * 74)
for name, m in all_metrics.items():
    mem_str = f"{m['peak_memory_mb']:.1f}" if m['peak_memory_mb'] > 0 else "N/A"
    print(f"{name:<22} {m['recon_mse']:>10.6f} {m['mean_step_ms']:>10.1f} {mem_str:>10} "
          f"{m['latent_mean_norm']:>10.4f} {m['latent_cov_frob']:>10.4f}")


Method                  Recon MSE    Step ms     Mem MB     ||mu||  ||S-I||_F
--------------------------------------------------------------------------
Spectral Wristband       0.008589      170.3      434.8     0.2153     0.6527
Pairwise Wristband       0.008787      168.3      435.6     0.2096     0.6689
WAE-MMD                  0.007395      170.0      451.9     0.3393     0.7244
VAE                      0.029334      166.1      436.1     0.3183     0.4852
SWAE                     0.006807      166.3      436.4     0.2859     1.1134
RFF-MMD                  0.006732      167.0      434.8     4.4225    52.4897


In [ ]:
# --- Training Loss Curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for name, m in all_metrics.items():
    if m["train_losses"]:
        ax.plot(range(1, len(m["train_losses"]) + 1), m["train_losses"], label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss (recon + reg)")
ax.set_title("Training Loss per Epoch")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
for name, m in all_metrics.items():
    if m["eval_recon_losses"]:
        ax.plot(range(1, len(m["eval_recon_losses"]) + 1), m["eval_recon_losses"], label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Eval Reconstruction Loss")
ax.set_title("Eval Reconstruction Loss per Epoch")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_curves.png")

## 5. Scaling Sweep — Step Time vs Batch Size

**This is the headline figure.** Log-log plot showing O(NdK) vs O(N²) scaling.
Sweep batch sizes and measure per-step time for each method.

In [ ]:
# Enable deterministic mode for reproducible timing
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SWEEP_BATCH_SIZES = [64, 128, 256, 512, 1024, 2048]
SWEEP_STEPS = 50       # steps per batch size
SWEEP_WARMUP = 5       # warmup steps (excluded from timing)

def time_model_at_batch_size(model, batch_size, num_steps, warmup):
    """Time `num_steps` forward+backward passes at a given batch size.
    Returns median step time in ms.
    """
    model.train()
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    step_times = []

    for step in range(warmup + num_steps):
        # Synthetic data (same distribution as MNIST, correct shape)
        x = torch.rand(batch_size, *INPUT_DIM, device=DEVICE)
        inputs = {"data": x}

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        out = model(inputs)
        out.loss.backward()
        opt.step()
        opt.zero_grad()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        dt = (time.perf_counter() - t0) * 1000  # ms

        if step >= warmup:
            step_times.append(dt)

    return np.median(step_times)


# Run sweep
scaling_results = {}  # {method_name: {batch_size: median_ms}}
for name, model in models.items():
    print(f"Sweeping: {name}", end="", flush=True)
    scaling_results[name] = {}
    for bs in SWEEP_BATCH_SIZES:
        # Re-create model to avoid state accumulation
        fresh_models = make_models(LATENT_DIM, INPUT_DIM)
        m = fresh_models[name].to(DEVICE)
        ms = time_model_at_batch_size(m, bs, SWEEP_STEPS, SWEEP_WARMUP)
        scaling_results[name][bs] = ms
        print(f"  N={bs}: {ms:.1f}ms", end="", flush=True)
    print()

# Restore fast mode for any subsequent cells
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

print("\nDone.")

In [ ]:
# --- Scaling Plot (log-log) ---
fig, ax = plt.subplots(figsize=(8, 6))

# Style: our method bold, baselines thinner
STYLE = {
    "Spectral Wristband": dict(color="C0", lw=2.5, marker="o", ms=8, zorder=10),
    "Pairwise Wristband": dict(color="C1", lw=2.0, marker="s", ms=7, zorder=9),
    "WAE-MMD":            dict(color="C2", lw=1.5, marker="^", ms=6),
    "VAE":                dict(color="C3", lw=1.5, marker="d", ms=6),
    "SWAE":               dict(color="C4", lw=1.5, marker="v", ms=6),
    "RFF-MMD":            dict(color="C5", lw=1.5, marker="P", ms=6),
}

for name, bs_dict in scaling_results.items():
    bs_list = sorted(bs_dict.keys())
    times = [bs_dict[bs] for bs in bs_list]
    style = STYLE.get(name, {})
    ax.plot(bs_list, times, label=name, **style)

ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("Batch size N", fontsize=12)
ax.set_ylabel("Step time (ms)", fontsize=12)
ax.set_title("Computational Scaling: Step Time vs Batch Size", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, which="both")
ax.set_xticks(SWEEP_BATCH_SIZES)
ax.set_xticklabels([str(b) for b in SWEEP_BATCH_SIZES])

plt.tight_layout()
plt.savefig("scaling_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: scaling_plot.png")

## 6. FID Evaluation

Generate samples from each model by decoding z ~ N(0,I), then compute FID against real data using `pytorch-fid`.

In [ ]:
import subprocess
from pathlib import Path
from torchvision.utils import save_image

FID_N_SAMPLES = 10000
FID_REAL_DIR = Path("./fid_real")
FID_GEN_DIR_BASE = Path("./fid_gen")

def save_images_for_fid(images_tensor, out_dir: Path, prefix="img"):
    """Save tensor images as PNGs for pytorch-fid."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for i in range(images_tensor.shape[0]):
        save_image(images_tensor[i], out_dir / f"{prefix}_{i:05d}.png")

# Save real images (once)
if not FID_REAL_DIR.exists():
    print("Saving real images for FID...")
    real_tensor = eval_data.imgs[:FID_N_SAMPLES].float()
    save_images_for_fid(real_tensor, FID_REAL_DIR)
    print(f"  Saved {FID_N_SAMPLES} real images to {FID_REAL_DIR}")

# Generate and save samples from each model, compute FID
fid_scores = {}
for name, model_obj in models.items():
    print(f"\nComputing FID: {name}")
    model_obj.eval()
    model_obj.to(DEVICE)
    gen_dir = FID_GEN_DIR_BASE / name.replace(" ", "_")

    with torch.no_grad():
        # Sample z ~ N(0, I), decode
        generated = []
        remaining = FID_N_SAMPLES
        while remaining > 0:
            bs = min(BATCH_SIZE, remaining)
            z = torch.randn(bs, LATENT_DIM, device=DEVICE)
            x_gen = model_obj.decoder(z)["reconstruction"]
            generated.append(x_gen.cpu().clamp(0, 1))
            remaining -= bs
        generated = torch.cat(generated, dim=0)

    save_images_for_fid(generated, gen_dir)

    # Run pytorch-fid
    result = subprocess.run(
        ["python", "-m", "pytorch_fid", str(FID_REAL_DIR), str(gen_dir), "--batch-size", "256"],
        capture_output=True, text=True,
    )
    # Parse FID from output: "FID:  123.45"
    fid_line = [l for l in result.stdout.strip().split("\n") if "FID" in l]
    if fid_line:
        fid_val = float(fid_line[-1].split()[-1])
        fid_scores[name] = fid_val
        print(f"  FID = {fid_val:.2f}")
    else:
        print(f"  FID computation failed: {result.stderr[:200]}")
        fid_scores[name] = float("nan")

print("\n--- FID Scores ---")
for name, fid in sorted(fid_scores.items(), key=lambda x: x[1]):
    print(f"  {name:<22} {fid:.2f}")

## 7. Summary Table

All metrics combined: FID, Recon MSE, Step Time, Peak Memory, Latent Diagnostics.

In [ ]:
# --- Full Summary Table ---
header = f"{'Method':<22} {'FID':>8} {'MSE':>10} {'ms/step':>10} {'Mem MB':>10} {'||mu||':>8} {'||S-I||':>8}"
print(f"\n{header}")
print("=" * len(header))

for name in models.keys():
    m = all_metrics[name]
    fid = fid_scores.get(name, float("nan"))
    mem = f"{m['peak_memory_mb']:.1f}" if m['peak_memory_mb'] > 0 else "N/A"
    print(f"{name:<22} {fid:>8.1f} {m['recon_mse']:>10.6f} {m['mean_step_ms']:>10.1f} "
          f"{mem:>10} {m['latent_mean_norm']:>8.4f} {m['latent_cov_frob']:>8.4f}")

# --- Bar chart of key metrics ---
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
names = list(models.keys())
colors = ["C0", "C1", "C2", "C3", "C4", "C5"]

# FID
ax = axes[0]
vals = [fid_scores.get(n, 0) for n in names]
ax.barh(names, vals, color=colors)
ax.set_xlabel("FID (lower is better)")
ax.set_title("FID")
ax.invert_yaxis()

# Recon MSE
ax = axes[1]
vals = [all_metrics[n]["recon_mse"] for n in names]
ax.barh(names, vals, color=colors)
ax.set_xlabel("Reconstruction MSE")
ax.set_title("Recon MSE")
ax.invert_yaxis()

# Step time
ax = axes[2]
vals = [all_metrics[n]["mean_step_ms"] for n in names]
ax.barh(names, vals, color=colors)
ax.set_xlabel("Step time (ms)")
ax.set_title("Step Time")
ax.invert_yaxis()

# Prior matching
ax = axes[3]
vals = [all_metrics[n]["latent_cov_frob"] for n in names]
ax.barh(names, vals, color=colors)
ax.set_xlabel("||Sigma - I||_F")
ax.set_title("Prior Matching (Covariance)")
ax.invert_yaxis()

plt.suptitle(f"Benchmark Results — {DATASET.upper()}, d={LATENT_DIM}, {NUM_EPOCHS} epochs",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("summary_bars.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: summary_bars.png")